# Day 07 · 開放協定：MCP 與 OpenAPI 整合

> 第二部・裝備升級　|　🔀 需要兩個程序（notebook 自動起停）

**前置需求**：🔑 需要 Gemini API 金鑰、📦 需要 `mcp` 套件（`google-adk[mcp]`）

**對應文章**：`Day 07 - 開放協定：MCP 與 OpenAPI 整合.md`

## 今天要學會

1. 用 `McpToolset` 把 ADK 當成 MCP **client**
2. 把 ADK agent 包成 MCP **server** 給別人用
3. 用 `OpenAPIToolset` 從 spec 自動生工具
4. ⚠️ 避開那個「本地會過、部署會炸」的同步定義陷阱

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


In [2]:
# 本日需要 mcp 套件
try:
    import mcp

    print("✅ mcp 已安裝")
except ImportError:
    print("❌ 請執行：uv sync（pyproject 已含 google-adk[mcp]）")

✅ mcp 已安裝


## 1. MCP 是協定，不是函式庫

這是最容易誤解的一點。**MCP（Model Context Protocol）本身不做任何事**，
它只定義「一個程序怎麼把工具描述給另一個程序」。

```
  ┌──────────────┐   JSON-RPC over stdio/HTTP   ┌──────────────┐
  │  ADK agent   │ ◄──────────────────────────► │  MCP server  │
  │ （client）    │   list_tools / call_tool     │ （別人寫的）  │
  └──────────────┘                              └──────────────┘
```

ADK 兩邊都能當：

| 角色 | 用什麼 | 情境 |
|---|---|---|
| **Client** | `McpToolset` | 用別人做好的工具（GitHub、資料庫、檔案系統…） |
| **Server** | `to_mcp_server` / 手寫 `Server` | 把你的 agent 能力開放給別人 |

## 2. 先寫一個 MCP server

為了讓本日完全自給自足，我們自己寫一個 server 再自己連。
用 `%%writefile` 把它寫成獨立檔案——因為它必須跑在**另一個程序**。

In [3]:
import shutil
import sys
from pathlib import Path

WORK = Path.cwd() / "_day07"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()
SERVER = WORK / "inventory_server.py"

SERVER.write_text('''"""一個最小的 MCP server：把 ADK 的 FunctionTool 用 MCP 協定開放出去。"""
import asyncio
import json

import mcp.server.stdio
from mcp import types as mcp_types
from mcp.server.lowlevel import NotificationOptions, Server
from mcp.server.models import InitializationOptions

from google.adk.tools import FunctionTool
from google.adk.tools.mcp_tool import adk_to_mcp_tool_type

_INVENTORY = {"A-100": 12, "A-200": 0, "B-300": 47}


def check_stock(sku: str) -> dict:
    """查詢商品庫存數量。

    Args:
        sku: 商品編號，例如 'A-100'。
    """
    qty = _INVENTORY.get(sku)
    if qty is None:
        return {"error": f"查無此商品：{sku}", "available": list(_INVENTORY)}
    return {"sku": sku, "quantity": qty, "in_stock": qty > 0}


def list_skus() -> dict:
    """列出所有商品編號。"""
    return {"skus": list(_INVENTORY)}


# 1) 用 ADK 的 FunctionTool 定義工具（沿用 Day 06 的寫法）
TOOLS = {t.name: t for t in [FunctionTool(func=check_stock), FunctionTool(func=list_skus)]}

app = Server("inventory-mcp-server")


# 2) 告訴 MCP client「我有哪些工具」——adk_to_mcp_tool_type 負責格式轉換
@app.list_tools()
async def list_tools() -> list[mcp_types.Tool]:
    return [adk_to_mcp_tool_type(t) for t in TOOLS.values()]


# 3) 實際被呼叫時，轉交給 ADK 的工具執行
@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[mcp_types.TextContent]:
    tool = TOOLS.get(name)
    if tool is None:
        payload = {"error": f"no such tool: {name}"}
    else:
        payload = await tool.run_async(args=arguments, tool_context=None)
    return [mcp_types.TextContent(type="text", text=json.dumps(payload, ensure_ascii=False))]


async def main():
    async with mcp.server.stdio.stdio_server() as (reader, writer):
        await app.run(
            reader,
            writer,
            InitializationOptions(
                server_name=app.name,
                server_version="0.1.0",
                capabilities=app.get_capabilities(
                    notification_options=NotificationOptions(),
                    experimental_capabilities={},
                ),
            ),
        )


if __name__ == "__main__":
    asyncio.run(main())
''', encoding="utf-8")

print(f"已寫出 {SERVER.name}（{SERVER.stat().st_size} bytes）")

已寫出 inventory_server.py（2265 bytes）


三個步驟很清楚：

1. 用 ADK 的 `FunctionTool` 定義工具（跟 Day 06 完全一樣）
2. `@app.list_tools()` 回報有哪些工具——`adk_to_mcp_tool_type()` 負責格式轉換
3. `@app.call_tool()` 被呼叫時轉交給 ADK 的工具執行

**你的工具程式碼一行都不用改**，只是多包一層協定。

## 3. 用 `McpToolset` 連上去

`StdioConnectionParams` 會**自動幫你把 server 當子程序啟動**，
你不用先手動開一個 terminal。

In [4]:
from mcp import StdioServerParameters

from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams

toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,          # 用目前這個 venv 的 python
            args=[str(SERVER)],
        ),
        timeout=30,
    )
)

# 先問問看它有什麼工具（這一步就會啟動 server 子程序）
tools = await toolset.get_tools()
print("MCP server 提供的工具:")
for t in tools:
    print(f"  • {t.name}: {t.description.splitlines()[0]}")

MCP server 提供的工具:
  • check_stock: 查詢商品庫存數量。
  • list_skus: 列出所有商品編號。


In [5]:
agent = LlmAgent(
    name="inventory_agent",
    model=get_model(),
    instruction="你是庫存助理。一律呼叫工具查詢，用繁體中文簡短回答。",
    tools=[toolset],          # ← 整個 toolset 丟進去，跟 Day 06 的 BaseToolset 一樣
)

print(await run_once(agent, "A-200 還有貨嗎？", trace=True))

  🔧 [inventory_agent] 呼叫 check_stock({'sku': 'A-200'})
  ↩️  [inventory_agent] check_stock 回傳 {'content': [{'type': 'text', 'text': '{"sku": "A-200", "quantity": 0, "in_stock": false}'}], 'isError': False}


  💬 [inventory_agent] A-200 目前沒有貨（庫存為 0）。
A-200 目前沒有貨（庫存為 0）。


### 📌 注意回傳值的形狀跟一般工具不一樣

看上面 trace 的 `function_response`——它不是你在 server 裡 return 的那個 dict，
而是被 MCP 包了一層：

```
{'content': [{'type': 'text', 'text': '{"sku": "A-200", ...}'}], 'isError': False}
```

這是 MCP 協定規定的信封格式（`content` 陣列 + `isError` 旗標）。
模型看得懂，但**如果你在 callback 或測試裡要解析工具結果，要多剝一層**。

In [6]:
result = await tools[0].run_async(
    args={"sku": "B-300"}, tool_context=None
)
print("原始回傳:", result)
print()
import json

inner = json.loads(result["content"][0]["text"])
print("剝掉信封之後:", inner)

原始回傳: {'content': [{'type': 'text', 'text': '{"sku": "B-300", "quantity": 47, "in_stock": true}'}], 'isError': False}

剝掉信封之後: {'sku': 'B-300', 'quantity': 47, 'in_stock': True}


## 4. ⚠️ 本日最重要的一節：部署會炸的同步定義規則

原文有提到但沒展開。這個坑的特徵是：

> **本地 `adk web` 完全正常，一部署就掛掉。**

原因是部署流程需要**同步地**載入你的 agent 模組來做檢查。
如果你的 agent 定義寫成 `async`，本地開發時 ADK 會幫你 await，
但部署工具沒有 event loop 可用。

In [7]:
BAD = WORK / "bad_agent.py"
BAD.write_text('''"""❌ 錯誤示範：agent 定義寫成 async。"""
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset


async def create_agent():          # ← async！
    toolset = McpToolset(connection_params=...)
    tools = await toolset.get_tools()      # ← 在模組層 await
    return LlmAgent(name="bad", model="...", tools=tools)


root_agent = create_agent()        # ← 這是一個 coroutine，不是 Agent
''', encoding="utf-8")

GOOD = WORK / "good_agent.py"
GOOD.write_text('''"""✅ 正確示範：同步定義，把 toolset 直接交給 agent。"""
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams

# 重點：不要在這裡 await get_tools()。
# 把整個 toolset 交給 agent，ADK 會在「執行時」才去連線取工具。
root_agent = LlmAgent(
    name="good",
    model="gemini-flash-lite-latest",
    instruction="...",
    tools=[McpToolset(connection_params=StdioConnectionParams(server_params=...))],
)
''', encoding="utf-8")

for f in (BAD, GOOD):
    print("=" * 60)
    print(f.name)
    print(f.read_text(encoding="utf-8"))

bad_agent.py


"""❌ 錯誤示範：agent 定義寫成 async。"""
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset


async def create_agent():          # ← async！
    toolset = McpToolset(connection_params=...)
    tools = await toolset.get_tools()      # ← 在模組層 await
    return LlmAgent(name="bad", model="...", tools=tools)


root_agent = create_agent()        # ← 這是一個 coroutine，不是 Agent

good_agent.py
"""✅ 正確示範：同步定義，把 toolset 直接交給 agent。"""
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams

# 重點：不要在這裡 await get_tools()。
# 把整個 toolset 交給 agent，ADK 會在「執行時」才去連線取工具。
root_agent = LlmAgent(
    name="good",
    model="gemini-flash-lite-latest",
    instruction="...",
    tools=[McpToolset(connection_params=StdioConnectionParams(server_params=...))],
)



### 差別在哪

| | ❌ 錯誤寫法 | ✅ 正確寫法 |
|---|---|---|
| `root_agent` 的型別 | `coroutine` | `LlmAgent` |
| 什麼時候連 MCP server | **模組載入時** | **執行時** |
| 本地 `adk web` | 可能會過 | 會過 |
| `adk deploy` | **炸** | 過 |

實際驗證一下型別差異：

In [8]:
async def create_agent_async():
    return LlmAgent(name="x", model=get_model(), instruction="i")


coro = create_agent_async()      # 沒有 await
print("async 定義得到的 root_agent 型別:", type(coro).__name__)
print("它是 LlmAgent 嗎？", isinstance(coro, LlmAgent))
coro.close()                     # 收掉，避免 warning

sync_agent = LlmAgent(name="y", model=get_model(), instruction="i")
print("\n同步定義得到的 root_agent 型別:", type(sync_agent).__name__)
print("它是 LlmAgent 嗎？", isinstance(sync_agent, LlmAgent))

async 定義得到的 root_agent 型別: coroutine
它是 LlmAgent 嗎？ False

同步定義得到的 root_agent 型別: LlmAgent
它是 LlmAgent 嗎？ True


**記住這一條**：

> agent 模組的最外層只能有**同步**的定義。
> 需要 async 的東西（連線、取工具清單）交給 ADK 在執行時做。

## 5. OpenAPI：從 spec 自動生工具

如果對方提供的是傳統 REST API + OpenAPI spec，不需要 MCP——
`OpenAPIToolset` 會直接把每個 endpoint 變成一個工具。

In [9]:
from google.adk.tools.openapi_tool import OpenAPIToolset

SPEC = {
    "openapi": "3.0.0",
    "info": {"title": "Pet Store", "version": "1.0.0"},
    "servers": [{"url": "https://petstore.example.com/v1"}],
    "paths": {
        "/pets/{petId}": {
            "get": {
                "operationId": "getPetById",
                "summary": "依照 ID 查詢單一寵物的資料。",
                "parameters": [{
                    "name": "petId", "in": "path", "required": True,
                    "description": "寵物的唯一識別碼。",
                    "schema": {"type": "integer"},
                }],
                "responses": {"200": {"description": "成功"}},
            }
        },
        "/pets": {
            "get": {
                "operationId": "listPets",
                "summary": "列出所有寵物，可用 limit 限制數量。",
                "parameters": [{
                    "name": "limit", "in": "query", "required": False,
                    "description": "最多回傳幾筆。",
                    "schema": {"type": "integer"},
                }],
                "responses": {"200": {"description": "成功"}},
            }
        },
    },
}

openapi_toolset = OpenAPIToolset(spec_dict=SPEC)
api_tools = await openapi_toolset.get_tools()

print(f"從 spec 自動生出 {len(api_tools)} 個工具：")
for t in api_tools:
    print(f"  • {t.name}: {t.description}")

從 spec 自動生出 2 個工具：
  • get_pet_by_id: 依照 ID 查詢單一寵物的資料。
  • list_pets: 列出所有寵物，可用 limit 限制數量。


`operationId` 變成工具名稱、`summary` 變成工具說明、`parameters` 變成參數 schema。

**這代表：對方的 OpenAPI spec 寫得好不好，直接決定你的 agent 好不好用。**
`summary` 空白、`description` 缺漏的 spec，生出來的工具模型也不會用。

In [10]:
decl = api_tools[0]._get_declaration()
print("模型看到的 schema:")
print(json.dumps(decl.parameters_json_schema, indent=2, ensure_ascii=False))

模型看到的 schema:
{
  "properties": {
    "pet_id": {
      "type": "integer",
      "description": "寵物的唯一識別碼。"
    }
  },
  "required": [
    "pet_id"
  ],
  "title": "getPetById_Arguments",
  "type": "object"
}


## 6. MCP vs OpenAPI 怎麼選

| | MCP | OpenAPI |
|---|---|---|
| 為誰設計 | **給 LLM 用的** | 給程式用的 |
| 傳輸 | stdio / SSE / HTTP | HTTP |
| 生態 | 新，但成長很快 | 成熟、到處都有 |
| 動態性 | 可以在執行時改變工具清單 | spec 是靜態的 |
| 需要對方配合嗎 | 要，對方得寫 MCP server | **不用**，有 spec 就行 |
| 適合 | 本機工具、需要有狀態的整合 | 現成的第三方 REST API |

**實務建議**：對方已經有 OpenAPI spec 就用 OpenAPI，最省事；
需要存取本機資源（檔案、資料庫、瀏覽器）或要有狀態，才值得上 MCP。

In [11]:
# 收工：關掉 MCP 連線與子程序
await toolset.close()
await openapi_toolset.close()
shutil.rmtree(WORK, ignore_errors=True)
print("已關閉連線並清理工作目錄")

已關閉連線並清理工作目錄


> **記得 `close()`**。`McpToolset` 背後是一個子程序，不關的話 notebook
> 結束後它可能還活著。`BaseToolset.close()` 就是為這件事存在的（Day 06）。

## 7. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| `ModuleNotFoundError: No module named 'mcp'` | 沒裝 extras：`google-adk[mcp]` |
| 本地正常、`adk deploy` 失敗 | **agent 模組寫成 async**，改成同步定義 |
| 解析工具結果拿到奇怪的 dict | MCP 回傳是 `{'content': [...], 'isError': ...}` 信封，要剝一層 |
| notebook 關了子程序還在 | 忘了 `await toolset.close()` |
| OpenAPI 生出的工具模型不會用 | spec 的 `summary` / `description` 太空泛 |
| MCP server 起不來但沒錯誤訊息 | `command` 路徑錯了；先手動跑一次那個指令確認 |

## 8. 動手練習

1. 幫 `inventory_server.py` 加第三個工具（例如「補貨」），
   確認 client 端**完全不用改**就看得到新工具。
2. 故意把 `command` 改成不存在的路徑，觀察錯誤訊息長什麼樣。
3. 把 `SPEC` 裡 `getPetById` 的 `summary` 刪掉，
   重看 `t.description`，想想模型還敢不敢用這個工具。
4. 用 `tool_filter=["check_stock"]` 只暴露其中一個 MCP 工具。

## 本日回顧

- **MCP 是協定不是函式庫**；ADK 兩邊都能當（`McpToolset` 當 client、
  `adk_to_mcp_tool_type` + `Server` 當 server）。
- **`StdioConnectionParams` 會自動啟動 server 子程序**，記得 `close()`。
- **MCP 的工具回傳被包在 `{'content': [...], 'isError': ...}` 信封裡**，
  自己解析時要多剝一層。
- **⚠️ agent 模組最外層必須是同步定義**。寫成 async 會「本地過、部署炸」——
  這是 MCP 整合最常見的上線事故。
- **OpenAPI 適合現成 REST API**（不需對方配合），**MCP 適合本機／有狀態的整合**。

---
**下一天 → `../day08_grounding/`**